# External transfer: TCGA-trained model on a GEO cohort

This notebook trains an `E2MModel` on TCGA lung adenocarcinoma (`LUAD`) and applies it to **GSE31210** (Okayama et al.), an independent lung-adenocarcinoma microarray cohort the model has never seen, to demonstrate cross-cohort prediction end to end.

**Requirements:** network access and `GEOparse` for the GEO download:

```bash
pip install GEOparse
```

**Caveat:** GSE31210 is Affymetrix microarray data, on a different measurement scale from the TCGA STAR counts the model is trained on. This notebook shows the *mechanics* of external prediction — align genes by symbol, then predict. For quantitative transfer, the manuscript batch-corrects each external cohort against its TCGA training set (ComBat / rank normalization); that correction is **not** performed here.

In [ ]:
from pathlib import Path
import pandas as pd

from e2m import Dataset, E2MModel

DATA_DIR = Path("e2m_data")
RESULT_DIR = Path("results/external_gse31210")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Train the mutation model on TCGA-LUAD

This downloads and prepares TCGA-LUAD on the first run (cached under `DATA_DIR`) and fits the multitask network on every matched sample. TMB is not needed here, so it is skipped.

In [ ]:
tcga = Dataset.from_tcga(["LUAD"], data_dir=DATA_DIR, with_tmb=False)
model = E2MModel().fit(tcga)
len(tcga.expression), len(model.targets)

## 2. Download and prepare the external cohort

`GEOparse` downloads the series matrix and its platform annotation. We map probes to gene symbols (taking the first symbol of multi-mapping probes), average duplicate symbols, and return a samples-by-genes frame with the same orientation the model expects.

In [ ]:
def _symbol_column(annotation_table):
    for name in ("Gene Symbol", "Gene symbol", "GENE_SYMBOL", "Symbol", "gene_assignment"):
        if name in annotation_table.columns:
            return name
    raise ValueError(
        f"No gene-symbol column in the platform annotation: {list(annotation_table.columns)}"
    )


def load_gse31210(cache_dir):
    """Download GSE31210, map probes to symbols, return a samples-by-genes frame."""
    import GEOparse

    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    gse = GEOparse.get_GEO(geo="GSE31210", destdir=str(cache_dir), silent=True)
    probes = gse.pivot_samples("VALUE")  # probes x samples

    platform = list(gse.gpls.values())[0]
    annotation = platform.table.set_index("ID")
    symbols = annotation[_symbol_column(annotation)].reindex(probes.index).astype(str)
    symbols = symbols.str.split(r"\s*///\s*").str[0].str.strip()

    keep = symbols.notna() & ~symbols.isin({"", "nan", "---"})
    probes = probes.loc[keep]
    probes.index = symbols[keep].values
    expression = probes.apply(pd.to_numeric, errors="coerce").groupby(level=0).mean().T
    expression.index.name = "sample"
    return expression

In [ ]:
external = load_gse31210(DATA_DIR / "geo")
external.shape

## 3. Predict mutation probabilities on the external cohort

Input genes are aligned to the model's training features by symbol. We set `min_feature_overlap=0.0` because a microarray covers only part of the protein-coding transcriptome; missing features are filled with their training means. Because the scales differ and no batch correction is applied, read the numbers as a mechanics demonstration rather than a calibrated result.

In [ ]:
probabilities = model.predict(external, min_feature_overlap=0.0)
probabilities.to_csv(RESULT_DIR / "gse31210_probabilities.csv")
probabilities.iloc[:5, :8]

In [ ]:
probabilities.mean().sort_values(ascending=False).head(10)

## 4. Evaluate against the reported EGFR and KRAS status

GSE31210 reports a `gene alteration status` for every tumor, so two of the model's targets can be scored on this cohort. The field takes four values across the 226 tumors: `EGFR mutation +` (127), `EGFR/KRAS/ALK -` (68), `KRAS mutation +` (20), and `ALK-fusion +` (11). A tumor is EGFR-positive only under the first value and KRAS-positive only under the third; every other tested tumor is a negative for that gene.

A few limits worth keeping in mind:

- ALK is not scored. The reported event is a rearrangement, not a gene-level nonsilent mutation, so it is not one of the model's targets.
- The labels come from the study's clinical assay rather than MC3 calls, and that assay covered only these three genes. A negative here means "not detected by that assay", which is weaker than a negative MC3 call.
- EGFR is mutated in 56% of this cohort against roughly 14% in TCGA-LUAD, because GSE31210 is a Japanese, never-smoker-enriched series. Read the ranking metrics rather than the raw probabilities: the model learned the TCGA prevalence and is not calibrated to this one.

The 20 normal-lung samples in the series carry no alteration status and drop out of the intersection below.

In [ ]:
def load_alteration_status(cache_dir):
    """EGFR / KRAS status reported for each GSE31210 tumor, from the cached download."""
    import GEOparse

    gse = GEOparse.get_GEO(geo="GSE31210", destdir=str(Path(cache_dir)), silent=True)
    reported = {}
    for name, gsm in gse.gsms.items():
        for field in gsm.metadata.get("characteristics_ch1", []):
            key, _, value = field.partition(":")
            if key.strip() == "gene alteration status":
                reported[name] = value.strip()

    status = pd.Series(reported, name="alteration").sort_index()
    return pd.DataFrame(
        {
            "EGFR": (status == "EGFR mutation +").astype(int),
            "KRAS": (status == "KRAS mutation +").astype(int),
        },
        index=status.index,
    )


status = load_alteration_status(DATA_DIR / "geo")
status.sum()

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

from e2m.metrics import normalized_auprc

scored = status.index.intersection(probabilities.index)

rows = []
for target in ("EGFR", "KRAS"):
    truth = status.loc[scored, target]
    probability = probabilities.loc[scored, target]
    prevalence = float(truth.mean())
    auprc = float(average_precision_score(truth, probability))
    rows.append(
        {
            "target": target,
            "n_samples": len(scored),
            "n_positive": int(truth.sum()),
            "prevalence": prevalence,
            "auprc": auprc,
            "normalized_auprc": normalized_auprc(auprc, prevalence),
            "roc_auc": float(roc_auc_score(truth, probability)),
        }
    )

external_metrics = pd.DataFrame(rows).set_index("target")
external_metrics.to_csv(RESULT_DIR / "gse31210_external_metrics.csv")
external_metrics.round(3)

`normalized_auprc` is the same quantity the cross-validation tables report, `(AUPRC - prevalence) / (1 - prevalence)`, where 0 is chance at the given prevalence and 1 is a perfect ranking. Threshold metrics such as accuracy, F1, and MCC are left out on purpose: a cutoff chosen at TCGA prevalence does not carry over to a cohort with four times the EGFR rate.

Whatever lands above chance here comes from expression alone, across a platform change and without batch correction, so treat it as a floor. Correcting the external cohort against the TCGA training set, as the manuscript does, is the next step if you want a number to quote.